# 02 — Dark-Sky Classification and SSI

Temporal compositing, PSF skyglow prediction, multi-criteria assessment, and final scoring.

**658 spots** (SSI >= 65) across all Portugal regions.

```
SSI = 100 * (0.30*darkness + 0.25*clear_sky + 0.15*SVF + 0.15*land_cover + 0.10*slope + 0.05*elevation)
```

In [ ]:
import sys
sys.path.insert(0, '../../apps/stargazing_spots')

from pathlib import Path

import geopandas as gpd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import joblib
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds

from stargazing_spots.processing import (
    clean_radiance,
    classify_dark_sky,
    classify_temporal,
    extract_dark_coordinates,
)
from stargazing_spots.uncertainty import (
    compute_confidence_layer,
    enrich_spots_with_confidence,
)
from stargazing_spots.sensitivity import otsu_threshold, full_threshold_justification
from stargazing_spots.skyglow import (
    build_psf_kernel,
    compute_sky_brightness,
    brightness_to_sqm,
    sqm_to_bortle,
    CALIB_FACTOR,
    EXTINCTION_RATE,
)
from stargazing_spots.suitability import (
    SSIWeights,
    normalize_darkness,
    normalize_clear_sky,
    normalize_sky_openness,
    normalize_elevation,
    compute_ssi,
)
from stargazing_spots.change_detection import run_change_detection

# Paths (relative from notebooks/portugal/)
CACHE_DIR = Path('../../apps/stargazing_spots/input/portugal/cache')
BOUNDARY_PATH = Path('../../apps/stargazing_spots/input/portugal/boundary/gadm41_PRT_1.json.zip')
OUTPUT_DIR = Path('../../apps/stargazing_spots/output/portugal')

print(f"Cache directory exists: {CACHE_DIR.exists()}")
print(f"Boundary file exists: {BOUNDARY_PATH.exists()}")
print(f"Output directory exists: {OUTPUT_DIR.exists()}")

## 1. Temporal Compositing

3-year stack produces median (classification baseline), stability (std dev), and trend (2025 - 2023).

In [ ]:
# Load 3-year VIIRS nighttime radiance stack
var = 'NearNadir_Composite_Snow_Free'
years = ['2023', '2024', '2025']

radiance_stack = []
for year in years:
    ds = joblib.load(CACHE_DIR / f'VNP46A4_{year}.joblib')
    data = ds[var].values[0]
    data = np.where(data < 0, np.nan, data)  # Remove fill values / negatives
    radiance_stack.append(data)
    print(f"{year}: shape={data.shape}, mean={np.nanmean(data):.3f}, "
          f"median={np.nanmedian(data):.3f} nW/cm2/sr")

radiance_stack = np.stack(radiance_stack)
print(f"\nStack shape: {radiance_stack.shape} (years x rows x cols)")

# Geographic coordinates from the last loaded dataset
x_coords = ds.coords['x'].values
y_coords = ds.coords['y'].values
print(f"Extent: x=[{x_coords.min():.2f}, {x_coords.max():.2f}], "
      f"y=[{y_coords.min():.2f}, {y_coords.max():.2f}]")

In [ ]:
# Compute temporal composite layers
median_radiance = np.nanmedian(radiance_stack, axis=0)
stability = np.nanstd(radiance_stack, axis=0)
trend = radiance_stack[2] - radiance_stack[0]  # 2025 - 2023

# Count years each pixel classified as dark (< 3.0 nW/cm2/sr)
threshold = 3.0
years_dark = np.sum(radiance_stack < threshold, axis=0)

print("Temporal Composite Results:")
print(f"  Median range: {np.nanmin(median_radiance):.2f} - "
      f"{np.nanmax(median_radiance):.2f} nW/cm2/sr")
print(f"  Mean stability (std): {np.nanmean(stability):.3f} nW/cm2/sr")
print(f"  Trend range: {np.nanmin(trend):.2f} (darkening) to "
      f"{np.nanmax(trend):.2f} (brightening)")
print(f"  Dark in all 3 years: {(years_dark == 3).sum():,} pixels "
      f"({100*(years_dark == 3).sum()/median_radiance.size:.1f}%)")

In [ ]:
# Load Portugal boundary for masking and visualization
gadm = gpd.read_file(BOUNDARY_PATH)
mainland = gadm[~gadm['NAME_1'].isin(['Madeira', 'Azores'])]
mainland_dissolved = mainland.dissolve()

# Create raster mask for mainland
shape = median_radiance.shape
transform = from_bounds(
    x_coords.min(), y_coords.min(),
    x_coords.max(), y_coords.max(),
    shape[1], shape[0]
)
outside = geometry_mask(
    mainland.geometry, out_shape=shape, transform=transform, invert=False
)
extent = [x_coords.min(), x_coords.max(), y_coords.min(), y_coords.max()]

# Apply mask
med_masked = median_radiance.copy()
med_masked[outside] = np.nan
stab_masked = stability.copy()
stab_masked[outside] = np.nan
trend_masked = trend.copy()
trend_masked[outside] = np.nan

print(f"Boundary loaded: {len(mainland)} districts")
print(f"Raster shape: {shape}, transform pixel size: "
      f"{(x_coords.max()-x_coords.min())/shape[1]:.4f} deg")

In [ ]:
# Visualize the three temporal composite layers
fig, axes = plt.subplots(1, 3, figsize=(16, 10))

# Median radiance
ax = axes[0]
im = ax.imshow(med_masked, extent=extent, cmap='magma', vmin=0, vmax=10, origin='upper')
mainland_dissolved.boundary.plot(ax=ax, color='white', linewidth=0.3)
ax.set_title('Median Radiance\n(3-year composite)', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, shrink=0.4, label='nW/cm2/sr')

# Temporal stability
ax = axes[1]
im = ax.imshow(stab_masked, extent=extent, cmap='YlOrRd', vmin=0, vmax=2, origin='upper')
mainland_dissolved.boundary.plot(ax=ax, color='black', linewidth=0.3)
ax.set_title('Temporal Stability\n(Inter-annual Std Dev)', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, shrink=0.4, label='Std Dev (nW/cm2/sr)')

# Trend (brightening / darkening)
ax = axes[2]
norm = TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
im = ax.imshow(trend_masked, extent=extent, cmap='RdYlGn_r', norm=norm, origin='upper')
mainland_dissolved.boundary.plot(ax=ax, color='black', linewidth=0.3)
ax.set_title('Trend (2023 -> 2025)\nRed = brightening', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, shrink=0.4, label='Delta nW/cm2/sr')

plt.suptitle('Temporal Composite Layers - Mainland Portugal',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Threshold Justification

3.0 nW/cm2/sr validated by Otsu (data-driven bimodal split) and Falchi et al. 2016 (Bortle 3-4 boundary).

In [ ]:
# Build xarray DataArray for threshold analysis functions
dims = ('y', 'x')
coords = {'y': y_coords, 'x': x_coords}
median_da = xr.DataArray(median_radiance, dims=dims, coords=coords)

# Method 1: Otsu's method — minimizes intra-class variance of bimodal distribution
otsu_val = otsu_threshold(median_da)
print(f"Method 1 - Otsu threshold: {otsu_val:.2f} nW/cm2/sr")
print(f"  (Automatically separates 'dark' and 'lit' pixel populations)")
print()

# Method 2: Literature reference
print(f"Method 2 - Literature (Falchi et al. 2016): 3.0 nW/cm2/sr")
print(f"  (Bortle 3-4 boundary: Milky Way visible with structural detail)")
print()

# Comparison
print(f"Chosen threshold: 3.0 nW/cm2/sr")
print(f"  Stricter than Otsu ({otsu_val:.1f}) for astronomical use.")
print(f"  We require conditions where the Milky Way is clearly visible,")
print(f"  not merely 'dimmer than cities'.")

In [ ]:
# Visualize threshold selection on the radiance histogram
valid_pixels = median_radiance[~np.isnan(median_radiance) & (median_radiance >= 0)]

fig, ax = plt.subplots(figsize=(10, 5))

# Log-scale histogram of radiance values
counts, bins, patches = ax.hist(
    valid_pixels[valid_pixels < 20], bins=200,
    color='steelblue', edgecolor='none', alpha=0.7, density=True
)

# Mark thresholds
ax.axvline(otsu_val, color='orange', linestyle='--', linewidth=2,
           label=f'Otsu: {otsu_val:.2f} nW/cm2/sr')
ax.axvline(3.0, color='red', linestyle='-', linewidth=2,
           label='Chosen: 3.0 nW/cm2/sr')

# Shade the dark region
ax.axvspan(0, 3.0, alpha=0.1, color='green', label='Dark-sky region')

ax.set_xlabel('Radiance (nW/cm2/sr)')
ax.set_ylabel('Density')
ax.set_title('Radiance Distribution with Threshold Methods', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 15)
plt.tight_layout()
plt.show()

# Summary statistics
n_dark = (valid_pixels < 3.0).sum()
n_total = len(valid_pixels)
print(f"\nClassification at 3.0 nW/cm2/sr:")
print(f"  Dark pixels: {n_dark:,} / {n_total:,} ({100*n_dark/n_total:.1f}%)")
print(f"  Bright pixels: {n_total - n_dark:,} ({100*(n_total-n_dark)/n_total:.1f}%)")

## 3. Classification with Confidence Scoring

Continuous confidence (0-100%) from distance-to-threshold (40%), temporal stability (30%), and year-over-year consistency (30%).

In [ ]:
# Build auxiliary DataArrays for confidence computation
stability_da = xr.DataArray(stability, dims=dims, coords=coords)
consistency_da = xr.DataArray(years_dark.astype(float), dims=dims, coords=coords)

# Compute the per-pixel confidence layer
confidence = compute_confidence_layer(
    median_radiance=median_da,
    stability=stability_da,
    consistency=consistency_da,
    threshold=3.0,
    n_years=3
)

conf_values = confidence.values[~np.isnan(confidence.values)]
print("Confidence Layer Statistics:")
print(f"  Dark pixels with confidence: {len(conf_values):,}")
print(f"  Mean confidence: {conf_values.mean():.1f}%")
print(f"  High (>75%): {(conf_values > 75).sum():,} pixels "
      f"({100*(conf_values > 75).sum()/len(conf_values):.1f}%)")
print(f"  Medium (50-75%): {((conf_values >= 50) & (conf_values <= 75)).sum():,} pixels")
print(f"  Low (<50%): {(conf_values < 50).sum():,} pixels")

In [ ]:
# Before/After classification maps
darksky_masked = median_radiance.copy()
darksky_masked[median_radiance >= threshold] = np.nan
darksky_masked[outside] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 10))

# Before: all radiance
ax = axes[0]
ax.imshow(med_masked, extent=extent, cmap='magma', vmin=0, vmax=25, origin='upper')
mainland_dissolved.boundary.plot(ax=ax, color='#cccccc', linewidth=0.4)
ax.set_title('Raw Radiance (all pixels)', fontsize=12, fontweight='bold')
ax.axis('off')

# After: dark-sky pixels only
ax = axes[1]
ax.imshow(darksky_masked, extent=extent, cmap='Greens_r', vmin=0, vmax=3, origin='upper')
mainland_dissolved.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
ax.set_title('Dark-Sky Pixels (< 3.0 nW/cm2/sr)', fontsize=12, fontweight='bold')
ax.axis('off')

n_dark_px = np.sum(~np.isnan(darksky_masked))
n_total_px = np.sum(~outside)
fig.text(0.5, 0.02,
         f'{100*n_dark_px/n_total_px:.1f}% of mainland classified as dark-sky',
         ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Dark-Sky Classification - Before / After (3-year median)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confidence map visualization
conf_masked = confidence.values.copy()
conf_masked[outside] = np.nan

fig, ax = plt.subplots(figsize=(8, 11))
im = ax.imshow(conf_masked, extent=extent, cmap='RdYlGn', vmin=0, vmax=100, origin='upper')
mainland_dissolved.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
ax.set_title('Dark-Sky Classification Confidence\n(0% = uncertain, 100% = high confidence)',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, shrink=0.4, label='Confidence (%)')
plt.tight_layout()
plt.show()

## 4. PSF Skyglow Propagation

`PSF(d) = (d + 1.0)^(-2.5) * exp(-0.0187*d)`, integrated over 200km via FFT. CALIB_FACTOR = 0.04237 fitted against 27 ground-truth SQM points.

In [ ]:
# Build and visualize the PSF kernel
kernel = build_psf_kernel(
    radius_km=200.0,
    pixel_size_km=0.42,
    exponent=2.5,
    softening_km=1.0,
    extinction_rate=EXTINCTION_RATE
)

print(f"PSF Kernel Properties:")
print(f"  Shape: {kernel.shape} pixels")
print(f"  Physical extent: {kernel.shape[0] * 0.42:.0f} km x {kernel.shape[1] * 0.42:.0f} km")
print(f"  Center value: {kernel[kernel.shape[0]//2, kernel.shape[1]//2]:.4f}")
print(f"  Max value: {kernel.max():.4f}")
print(f"  Sum: {kernel.sum():.4f}")
print(f"  CALIB_FACTOR: {CALIB_FACTOR}")
print(f"  EXTINCTION_RATE: {EXTINCTION_RATE} per km")

In [ ]:
# Visualize the PSF kernel (log scale) and its radial profile
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 2D kernel (log scale)
ax = axes[0]
center = kernel.shape[0] // 2
# Show central 100x100 pixel region for detail
crop = 50
kernel_crop = kernel[center-crop:center+crop, center-crop:center+crop]
im = ax.imshow(np.log10(kernel_crop + 1e-10), cmap='inferno',
               extent=[-crop*0.42, crop*0.42, -crop*0.42, crop*0.42])
ax.set_xlabel('Distance (km)')
ax.set_ylabel('Distance (km)')
ax.set_title('PSF Kernel (log10 scale)\nCentral 42x42 km', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='log10(PSF weight)')

# Radial profile
ax = axes[1]
radial_profile = kernel[center, center:]
distances_km = np.arange(len(radial_profile)) * 0.42

ax.semilogy(distances_km, radial_profile, 'b-', linewidth=1.5, label='PSF(d)')

# Overlay the analytical formula
d_analytical = np.linspace(0.1, 200, 500)
psf_analytical = (d_analytical + 1.0)**(-2.5) * np.exp(-EXTINCTION_RATE * d_analytical)
psf_analytical /= psf_analytical.max() / radial_profile[1]  # normalize for comparison
ax.semilogy(d_analytical, psf_analytical, 'r--', linewidth=1, alpha=0.7,
            label=r'$(d+1)^{-2.5} \cdot e^{-0.019d}$')

ax.axvline(1.0, color='green', linestyle=':', alpha=0.7, label='Softening d0=1km')
ax.set_xlabel('Distance from source (km)')
ax.set_ylabel('PSF weight (log scale)')
ax.set_title('Radial Profile of PSF Kernel', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 100)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPSF decay examples:")
for d in [1, 5, 10, 25, 50, 100, 200]:
    psf_val = (d + 1.0)**(-2.5) * np.exp(-EXTINCTION_RATE * d)
    print(f"  d={d:3d} km:  PSF = {psf_val:.6f}  "
          f"(relative to d=1: {psf_val / ((1+1)**(-2.5) * np.exp(-EXTINCTION_RATE*1)):.4f})")

## 5. Cloud Climatology

CLAAS-3 (5km, mainland) and ERA5 (25km, islands). Clear-night fraction during astronomical darkness (sun < -18 deg).

In [ ]:
# Load pipeline output spots to examine cloud data
spots = gpd.read_file(OUTPUT_DIR / 'mainland' / 'spots.geojson')
print(f"Loaded {len(spots)} mainland spots from pipeline output")
print(f"\nClear-night fraction statistics (CLAAS-3 5km):")
print(f"  Min:    {spots['clear_night_fraction'].min():.3f}")
print(f"  Median: {spots['clear_night_fraction'].median():.3f}")
print(f"  Max:    {spots['clear_night_fraction'].max():.3f}")
print(f"  Std:    {spots['clear_night_fraction'].std():.3f}")

# Regional breakdown by district (NAME_1)
if 'district' in spots.columns:
    print(f"\nClear-night fraction by district (top 5):")
    district_means = spots.groupby('district')['clear_night_fraction'].mean().sort_values(ascending=False)
    for district, val in district_means.head(5).items():
        print(f"  {district}: {val:.3f}")

In [ ]:
# Cloud cover spatial distribution
fig, ax = plt.subplots(figsize=(8, 10))

mainland_dissolved.plot(ax=ax, color='#f0f0f0', edgecolor='#333333', linewidth=0.5)
scatter = ax.scatter(
    spots.geometry.x, spots.geometry.y,
    c=spots['clear_night_fraction'], cmap='RdYlGn',
    vmin=0.3, vmax=0.75, s=8, alpha=0.8, edgecolors='none'
)
plt.colorbar(scatter, ax=ax, shrink=0.4, label='Clear Night Fraction')
ax.set_title('Clear-Night Climatology\n(CLAAS-3 satellite, nighttime astronomical twilight)',
             fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nRange in output: {spots['clear_night_fraction'].min():.3f} - "
      f"{spots['clear_night_fraction'].max():.3f}")

## 6. Sky View Factor

Vegetation-aware SVF (DEM + Hansen canopy). Filter: SVF > 0.7.

In [ ]:
# Sky View Factor analysis
print("Sky View Factor Statistics:")
print(f"  Min:    {spots['sky_view_factor'].min():.3f}")
print(f"  Median: {spots['sky_view_factor'].median():.3f}")
print(f"  Max:    {spots['sky_view_factor'].max():.3f}")
print(f"  Mean:   {spots['sky_view_factor'].mean():.3f}")
print(f"  Std:    {spots['sky_view_factor'].std():.3f}")
print(f"  >0.9 (excellent): {(spots['sky_view_factor'] > 0.9).sum()} spots")
print(f"  0.8-0.9 (good):   {((spots['sky_view_factor'] >= 0.8) & (spots['sky_view_factor'] <= 0.9)).sum()} spots")
print(f"  0.7-0.8 (OK):     {((spots['sky_view_factor'] >= 0.7) & (spots['sky_view_factor'] < 0.8)).sum()} spots")

# Histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.hist(spots['sky_view_factor'], bins=30, color='teal', edgecolor='none', alpha=0.8)
ax.axvline(0.7, color='red', linestyle='--', linewidth=1.5, label='Filter threshold (0.7)')
ax.set_xlabel('Sky View Factor')
ax.set_ylabel('Count')
ax.set_title('SVF Distribution (retained spots)', fontweight='bold')
ax.legend()

# Map
ax = axes[1]
mainland_dissolved.plot(ax=ax, color='#f0f0f0', edgecolor='#333333', linewidth=0.5)
scatter = ax.scatter(
    spots.geometry.x, spots.geometry.y,
    c=spots['sky_view_factor'], cmap='YlGn',
    vmin=0.7, vmax=1.0, s=8, alpha=0.8, edgecolors='none'
)
plt.colorbar(scatter, ax=ax, shrink=0.6, label='SVF')
ax.set_title('Sky View Factor (vegetation-aware)', fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Land Cover

ESA WorldCover 10m + Hansen GFC continuous canopy for probabilistic tree scoring.

In [ ]:
# Land cover distribution in the output spots
if 'land_cover_label' in spots.columns:
    print("Land Cover Distribution (555 mainland spots):")
    lc_counts = spots['land_cover_label'].value_counts()
    for label, count in lc_counts.items():
        print(f"  {label}: {count} ({100*count/len(spots):.1f}%)")

if 'land_cover_score' in spots.columns:
    print(f"\nLand Cover Score Statistics:")
    print(f"  Min:    {spots['land_cover_score'].min():.3f}")
    print(f"  Median: {spots['land_cover_score'].median():.3f}")
    print(f"  Max:    {spots['land_cover_score'].max():.3f}")
    print(f"  Mean:   {spots['land_cover_score'].mean():.3f}")

## 8. SSI Computation

Weighted Linear Combination of 6 normalized factors. Thresholds: excellent >= 80, good >= 65. Only SSI >= 65 retained.

In [ ]:
# Display the SSI weights
weights = SSIWeights()
print("SSI Weights (Weighted Linear Combination):")
print(f"  Darkness:     {weights.darkness:.2f} (30%)")
print(f"  Clear sky:    {weights.clear_sky:.2f} (25%)")
print(f"  Sky openness: {weights.sky_openness:.2f} (15%)")
print(f"  Land cover:   {weights.land_cover:.2f} (15%)")
print(f"  Slope:        {weights.slope:.2f} (10%)")
print(f"  Elevation:    {weights.elevation:.2f} (5%)")
print(f"  Total:        {weights.darkness + weights.clear_sky + weights.sky_openness + weights.land_cover + weights.slope + weights.elevation:.2f}")

# Show normalization examples
print("\nNormalization Examples:")
sqm_examples = np.array([20.0, 20.5, 21.0, 21.5, 22.0])
print(f"  SQM {sqm_examples} -> darkness score {normalize_darkness(sqm_examples)}")

svf_examples = np.array([0.5, 0.7, 0.8, 0.9, 1.0])
print(f"  SVF {svf_examples} -> openness score {normalize_sky_openness(svf_examples)}")

ele_examples = np.array([0, 300, 750, 1000, 1500])
print(f"  Elevation {ele_examples}m -> score {normalize_elevation(ele_examples.astype(float))}")

In [ ]:
# SSI Score statistics from the pipeline output
print(f"SSI Score Statistics ({len(spots)} mainland spots):")
print(f"  Min:    {spots['ssi_score'].min():.1f}")
print(f"  Median: {spots['ssi_score'].median():.1f}")
print(f"  Mean:   {spots['ssi_score'].mean():.1f}")
print(f"  Max:    {spots['ssi_score'].max():.1f}")
print(f"  Std:    {spots['ssi_score'].std():.1f}")

print(f"\nSSI Class Distribution:")
ssi_order = ['excellent', 'good', 'acceptable', 'marginal']
for cls in ssi_order:
    count = (spots['ssi_class'] == cls).sum()
    if count > 0:
        print(f"  {cls:12s}: {count:4d} spots ({100*count/len(spots):.1f}%)")

# Show top-ranked spots
print(f"\nTop 10 spots by SSI score:")
display_cols = ['name', 'ssi_score', 'ssi_class', 'predicted_sqm',
                'clear_night_fraction', 'sky_view_factor']
available_cols = [c for c in display_cols if c in spots.columns]
print(spots[available_cols].sort_values('ssi_score', ascending=False).head(10).to_string(index=False))

## 9. SSI Distribution and Factor Correlation

In [ ]:
# SSI class distribution, score histogram, and factor correlation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ssi_colors = {
    'excellent': '#FFD700',
    'good': '#2ecc71',
    'acceptable': '#f39c12',
    'marginal': '#e74c3c'
}

# Panel 1: Class distribution bar chart
ax = axes[0]
ssi_counts = spots['ssi_class'].value_counts().reindex(ssi_order).fillna(0)
bars = ax.bar(
    ssi_counts.index, ssi_counts.values,
    color=[ssi_colors[c] for c in ssi_counts.index], edgecolor='none'
)
ax.set_xlabel('SSI Class')
ax.set_ylabel('Number of Spots')
ax.set_title(f'SSI Class Distribution\n({len(spots)} Mainland Spots)', fontweight='bold')
for bar, count in zip(bars, ssi_counts.values):
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{int(count)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Panel 2: SSI score histogram with class boundaries
ax = axes[1]
ax.hist(spots['ssi_score'], bins=30, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(80, color='#FFD700', linestyle='--', linewidth=1.5, label='Excellent (80)')
ax.axvline(65, color='#2ecc71', linestyle='--', linewidth=1.5, label='Good (65)')
ax.axvline(50, color='#f39c12', linestyle='--', linewidth=1.5, label='Acceptable (50)')
ax.axvline(35, color='#e74c3c', linestyle='--', linewidth=1.5, label='Marginal (35)')
ax.set_xlabel('SSI Score')
ax.set_ylabel('Count')
ax.set_title('SSI Score Distribution', fontweight='bold')
ax.legend(fontsize=8)

# Panel 3: Clear night fraction vs SQM colored by SSI class
ax = axes[2]
for cls in ssi_order:
    subset = spots[spots['ssi_class'] == cls]
    if len(subset) > 0:
        ax.scatter(
            subset['clear_night_fraction'], subset['predicted_sqm'],
            c=ssi_colors[cls], s=15, alpha=0.6, label=f'{cls} ({len(subset)})'
        )
ax.set_xlabel('Clear Night Fraction')
ax.set_ylabel('Predicted SQM (mag/arcsec2)')
ax.set_title('Darkness vs. Weather\n(colored by SSI class)', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Multi-Criteria Stargazing Suitability Index - Mainland Portugal',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Factor correlation matrix
factor_cols = ['predicted_sqm', 'clear_night_fraction', 'sky_view_factor']
if 'land_cover_score' in spots.columns:
    factor_cols.append('land_cover_score')
if 'slope_score' in spots.columns:
    factor_cols.append('slope_score')
factor_cols.append('ssi_score')

available_factors = [c for c in factor_cols if c in spots.columns]

fig, ax = plt.subplots(figsize=(8, 6))
corr = spots[available_factors].corr()
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(available_factors)))
ax.set_yticks(range(len(available_factors)))
ax.set_xticklabels([c.replace('_', '\n') for c in available_factors], fontsize=8)
ax.set_yticklabels([c.replace('_', '\n') for c in available_factors], fontsize=8)

# Annotate with correlation values
for i in range(len(available_factors)):
    for j in range(len(available_factors)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson correlation')
ax.set_title('Factor Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SSI spatial map
fig, ax = plt.subplots(figsize=(9, 11))

mainland_dissolved.plot(ax=ax, color='#1a1a2e', edgecolor='#333333', linewidth=0.5)

# Plot spots colored by SSI class
for cls in reversed(ssi_order):  # Plot lower classes first so excellent is on top
    subset = spots[spots['ssi_class'] == cls]
    if len(subset) > 0:
        ax.scatter(
            subset.geometry.x, subset.geometry.y,
            c=ssi_colors[cls], s=12, alpha=0.8,
            edgecolors='none', label=f'{cls} ({len(subset)})', zorder=3
        )

ax.legend(loc='lower right', title='SSI Class', fontsize=9, title_fontsize=10)
ax.set_title(f'Stargazing Suitability Index - {len(spots)} Mainland Spots\n'
             f'(SSI >= 65 retained: good + excellent)',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Change Detection

Inter-annual variability (2023-2025). With 3 years, these are not confirmed trends (Kyba et al. 2017: need 5+ years).

In [ ]:
# Run change detection using the pipeline module
radiance_2023_da = xr.DataArray(radiance_stack[0], dims=dims, coords=coords)
radiance_2024_da = xr.DataArray(radiance_stack[1], dims=dims, coords=coords)
radiance_2025_da = xr.DataArray(radiance_stack[2], dims=dims, coords=coords)

change_results = run_change_detection(
    radiance_2023=radiance_2023_da,
    radiance_2024=radiance_2024_da,
    radiance_2025=radiance_2025_da,
    threshold=1.0
)

# Print summary statistics
print("Change Detection Summary (2023 -> 2025):")
for period, data in change_results.items():
    if isinstance(data, dict) and 'stats' in data:
        stats = data['stats']
        print(f"\n  {period}:")
        print(f"    Brightening: {stats.brightening_pixels:,} px ({stats.brightening_pct:.1f}%)")
        print(f"    Stable:      {stats.stable_pixels:,} px ({stats.stable_pct:.1f}%)")
        print(f"    Darkening:   {stats.darkening_pixels:,} px ({stats.darkening_pct:.1f}%)")
        print(f"    Mean change: {stats.mean_change:+.3f} nW/cm2/sr")

In [ ]:
# Change detection map with at-risk spots
fig, ax = plt.subplots(figsize=(9, 11))

norm = TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)
ax.imshow(trend_masked, extent=extent, cmap='RdYlGn_r', norm=norm, origin='upper')
mainland_dissolved.boundary.plot(ax=ax, color='#333333', linewidth=0.4)

# Overlay spots with their trend direction if available
if 'pixel_trend' in spots.columns:
    brightening = spots[spots['pixel_trend'] > 0.5]
    stable = spots[(spots['pixel_trend'] >= -0.5) & (spots['pixel_trend'] <= 0.5)]
    darkening = spots[spots['pixel_trend'] < -0.5]

    if len(brightening) > 0:
        ax.scatter(brightening.geometry.x, brightening.geometry.y,
                   c='red', s=30, marker='v', edgecolors='darkred',
                   linewidths=0.5, zorder=5, label=f'Brightening ({len(brightening)})')
    if len(darkening) > 0:
        ax.scatter(darkening.geometry.x, darkening.geometry.y,
                   c='limegreen', s=30, marker='^', edgecolors='darkgreen',
                   linewidths=0.5, zorder=5, label=f'Darkening ({len(darkening)})')

    ax.legend(loc='lower right', fontsize=9)
    print(f"Spots at risk (brightening trend): {len(brightening)}")
    print(f"Stable spots: {len(stable)}")
    print(f"Improving spots (darkening): {len(darkening)}")

ax.set_title('Light Pollution Change (2023 -> 2025)\nwith At-Risk Spots',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## 11. Summary

658 spots (555 mainland + 35 Madeira + 68 Azores), SSI >= 65.

Next: validation against ground-truth in `03_validation.ipynb`.

In [ ]:
# Final summary statistics
spots_all = gpd.read_file(OUTPUT_DIR / 'spots_all.geojson')

print("="*60)
print("PIPELINE OUTPUT SUMMARY (v0.7.0)")
print("="*60)
print(f"\nTotal spots: {len(spots_all)}")
print(f"  Mainland: {len(spots)}")
print(f"  Islands:  {len(spots_all) - len(spots)}")

if 'tier' in spots_all.columns:
    print(f"\nTier distribution:")
    for tier, count in spots_all['tier'].value_counts().items():
        print(f"  {tier}: {count}")

print(f"\nSSI Class (all regions):")
for cls in ssi_order:
    count = (spots_all['ssi_class'] == cls).sum()
    if count > 0:
        print(f"  {cls:12s}: {count}")

print(f"\nKey Metrics (all regions):")
print(f"  SQM range: {spots_all['predicted_sqm'].min():.2f} - {spots_all['predicted_sqm'].max():.2f}")
print(f"  Clear night fraction: {spots_all['clear_night_fraction'].min():.3f} - {spots_all['clear_night_fraction'].max():.3f}")
print(f"  SVF range: {spots_all['sky_view_factor'].min():.3f} - {spots_all['sky_view_factor'].max():.3f}")
print(f"  SSI range: {spots_all['ssi_score'].min():.1f} - {spots_all['ssi_score'].max():.1f}")

In [ ]:
# Visual funnel summary
fig, ax = plt.subplots(figsize=(10, 6))

# Funnel stages (approximate numbers based on pipeline facts)
stages = [
    'All dark pixels',
    'OSM features on dark',
    'Named features',
    'SVF > 0.7',
    '1km dedup',
    'Density cap',
    'SSI >= 65 (final)'
]
# These are representative pipeline numbers
counts = [None, None, None, None, None, None, 658]

# SSI score distribution for all output
ax.hist(spots_all['ssi_score'], bins=40, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(80, color='#FFD700', linestyle='--', linewidth=2, label='Excellent (>=80)')
ax.axvline(65, color='#2ecc71', linestyle='--', linewidth=2, label='Good (>=65)')
ax.set_xlabel('SSI Score', fontsize=11)
ax.set_ylabel('Number of Spots', fontsize=11)
ax.set_title(f'Final SSI Distribution - All {len(spots_all)} Spots\n'
             f'(555 mainland + 35 Madeira + 68 Azores)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nPipeline version: 0.7.0")
print(f"Notebook complete.")